# Part 2: CNNs with hls4ml

This notebook applies the Part 1 hls4ml workflow to a small convolutional neural network for MNIST digit recognition.

The live path uses one prepared baseline CNN:

1. Load MNIST and a pre-trained CNN
2. Explain what changes for CNNs in hls4ml
3. Convert the CNN with `io_type='io_stream'`
4. Compare Keras and hls4ml bit-accurate emulation
5. Synthesize the baseline design and read the report
6. Discuss quantization and pruning as next optimization steps

## Setup

In [ ]:
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import subprocess

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score

%matplotlib inline

seed = 0
np.random.seed(seed)
tf.random.set_seed(seed)

def source_settings(settings_path):
    result = subprocess.run(
        ['bash', '-lc', f'source {settings_path} >/dev/null 2>&1 && env'],
        check=True, capture_output=True, text=True,
    )
    os.environ.update(
        line.split('=', 1) for line in result.stdout.splitlines() if '=' in line
    )

if 'vitis_hls' not in os.environ:
    os.environ['PATH'] = '/opt/Xilinx/Vitis_HLS/2024.1/bin:' + os.environ['PATH']

## Load MNIST

MNIST is a useful CNN example because the task is visually obvious: classify a 28x28 grayscale image as one digit from 0 to 9.

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

X_eval = np.ascontiguousarray(X_test[:3000])
y_eval = y_test[:3000]

print('Train:', X_train.shape, y_train_cat.shape)
print('Test:', X_test.shape, y_test_cat.shape)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(str(y_train[i]))
    ax.axis('off')
plt.tight_layout()

## Baseline CNN

For the tutorial we load a pre-trained CNN so that time is spent on conversion, emulation, synthesis, and report interpretation. The next cells also include the training path for regenerating the model.

### Define the Baseline Architecture

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Activation, Flatten, Dense, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1


def make_mnist_cnn():
    x = x_in = Input(shape=(28, 28, 1))

    x = Conv2D(8, (3, 3), kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001), use_bias=False, name='conv_0')(x)
    x = BatchNormalization(name='bn_conv_0')(x)
    x = Activation('relu', name='conv_act_0')(x)
    x = MaxPooling2D((2, 2), name='pool_0')(x)

    x = Conv2D(16, (3, 3), kernel_initializer='lecun_uniform', kernel_regularizer=l1(0.0001), use_bias=False, name='conv_1')(x)
    x = BatchNormalization(name='bn_conv_1')(x)
    x = Activation('relu', name='conv_act_1')(x)
    x = MaxPooling2D((2, 2), name='pool_1')(x)

    x = Flatten(name='flatten')(x)
    x = Dense(10, name='output_dense')(x)
    x_out = Activation('softmax', name='output_softmax')(x)

    return Model(inputs=[x_in], outputs=[x_out], name='mnist_baseline')

### Optional: Train the Baseline Model

Leave `TRAIN_MODEL = False` for the tutorial. Set it to `True` only if you want to regenerate `models/mnist_cnn_baseline.keras`.

In [ ]:
TRAIN_MODEL = False
baseline_model_path = Path('models/mnist_cnn_baseline.keras')

if TRAIN_MODEL:
    baseline_model_path.parent.mkdir(exist_ok=True)
    model = make_mnist_cnn()
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(
        X_train, y_train_cat,
        batch_size=256, epochs=10,
        validation_split=0.1, shuffle=True,
    )
    model.save(baseline_model_path)

### Load the Baseline Model

In [ ]:
baseline_model_path = Path('models/mnist_cnn_baseline.keras')
if not baseline_model_path.exists():
    raise FileNotFoundError(f'{baseline_model_path} not found. Provide the trained baseline model or set TRAIN_MODEL = True and run the training cell.')

model = load_model(baseline_model_path)
model.summary()
print('Model compute dtype:',model.compute_dtype)

### Check Layer Sizes

For the lowest latency, hls4ml can fully unroll small layers. If one layer is too large, use `Strategy='Resource'` or increase the reuse factor for that layer.

In [ ]:
for layer in model.layers:
    if layer.__class__.__name__ in ['Conv2D', 'Dense']:
        weights = layer.get_weights()
        if not weights:
            continue
        layer_size = int(np.prod(weights[0].shape))
        print(f'{layer.name}: {layer_size} weights')
        if layer_size > 4096:
            print('  WARNING: consider Resource strategy or a higher ReuseFactor for this layer')

## CNNs in hls4ml

The conversion flow is the same as Part 1, but CNNs need one important configuration change: use `io_type='io_stream'`.

**Use `io_type='io_stream'` when synthesizing convolutional neural networks with hls4ml.**

The CNN implementation in hls4ml is based on streams. In hardware, these streams become FIFO buffers. Shift registers keep the last `<kernel height - 1>` rows of pixels and maintain the current convolution window as pixels move through the design.

The animation below shows the idea: the input image streams in, the shift-register state tracks the convolution window, and the output image is produced as the window moves.

![The implementation of convolutional layers in hls4ml](images/conv2d_animation.gif)

For this small model we use `Strategy='Latency'`, which asks hls4ml to use more parallelism for lower latency. If a layer becomes too large for full unrolling, the usual options are to switch that layer to `Strategy='Resource'` or increase its `ReuseFactor`.

One practical detail: automatic precision can sometimes choose a very wide result type before the Softmax lookup table. We explicitly constrain the final dense result precision to keep the Softmax implementation reasonable.

## Convert the Baseline CNN

This is the Part 1 conversion pattern with the CNN-specific streaming I/O setting.

In [ ]:
import hls4ml
import plotting

hls_config = hls4ml.utils.config_from_keras_model(
    model, granularity='name', backend='Vitis', default_precision='ap_fixed<16,6>'
)
hls_config['Model']['Strategy'] = 'Latency'

for layer_name in ['output_dense','output_dense_linear']:
    if layer_name in hls_config.get('LayerName', {}):
        hls_config['LayerName'][layer_name]['Precision']['result'] = 'fixed<16,6,RND,SAT>'

plotting.print_dict(hls_config)

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=hls_config,
    backend='Vitis',
    output_dir='mnist_cnn_baseline',
    part='xczu9eg-ffvb1156-2-e',
    io_type='io_stream',
)

### Visualize the HLS Model

The graph shows the converted model, tensor shapes, and assigned precisions.

In [ ]:
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file=None)

## Bit-Accurate Emulation

Before synthesis, compare the floating-point Keras model with hls4ml fixed-point emulation. This is the fastest way to catch precision problems.

In [ ]:
hls_model.compile()

y_keras = model.predict(X_eval)
y_hls = hls_model.predict(X_eval)

keras_labels = np.argmax(y_keras, axis=1)
hls_labels = np.argmax(y_hls, axis=1)

print('Keras  accuracy: {:.4f}'.format(accuracy_score(y_eval, keras_labels)))
print('hls4ml accuracy: {:.4f}'.format(accuracy_score(y_eval, hls_labels)))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_eval, keras_labels, ax=axes[0], colorbar=False, values_format='d')
axes[0].set_title('Keras')
ConfusionMatrixDisplay.from_predictions(y_eval, hls_labels, ax=axes[1], colorbar=False, values_format='d')
axes[1].set_title('hls4ml')
plt.tight_layout()

## Synthesize the Baseline

C-synthesis produces the latency, initiation interval, and resource estimates for the ZCU102 target.

In [ ]:
hls_model.build(csim=False)

### Read the Synthesis Report

Focus on latency, initiation interval, DSPs, LUTs, FFs, and BRAM. These numbers are the hardware cost of the baseline design.

In [ ]:
hls4ml.report.read_vivado_report('mnist_cnn_baseline')

## Optimizing for resources: Quantization

Quantization reduces the number of bits used for weights, activations, and intermediate results. Smaller arithmetic can reduce DSP use, LUT use, memory, or routing pressure depending on the model and target.

For lower precision, quantization-aware training with QKeras is usually more robust than simply lowering hls4ml precision after training.

### Define the QKeras Model
QKeras provides quantized versions of Keras layers. Here we replace floating-point convolution and dense layers with quantized layers so the model learns to operate with reduced precision during training.

We use 6-bit quantizers for weights, biases, and ReLU activations. In QKeras, the bit count does not include the sign bit in the same way hls4ml reports fixed-point types, so the hls4ml precision may appear one bit wider than the QKeras setting.

For CNNs, `QConv2DBatchnorm` combines convolution and batch normalization into one quantized layer. This is useful for hls4ml because batch normalization can be folded into the convolution parameters for inference.

In [ ]:
def make_qkeras_mnist_cnn():
    from qkeras import QActivation, QConv2DBatchnorm, QDense

    x = x_in = Input(shape=(28, 28, 1))

    x = QConv2DBatchnorm(
        8, (3, 3),
        kernel_quantizer='quantized_bits(6,0,alpha=1)',
        bias_quantizer='quantized_bits(6,0,alpha=1)',
        kernel_initializer='lecun_uniform',
        kernel_regularizer=l1(0.0001),
        use_bias=True,
        name='fused_convbn_0',
    )(x)
    x = QActivation('quantized_relu(6)', name='conv_act_0')(x)
    x = MaxPooling2D((2, 2), name='pool_0')(x)

    x = QConv2DBatchnorm(
        16, (3, 3),
        kernel_quantizer='quantized_bits(6,0,alpha=1)',
        bias_quantizer='quantized_bits(6,0,alpha=1)',
        kernel_initializer='lecun_uniform',
        kernel_regularizer=l1(0.0001),
        use_bias=True,
        name='fused_convbn_1',
    )(x)
    x = QActivation('quantized_relu(6)', name='conv_act_1')(x)
    x = MaxPooling2D((2, 2), name='pool_1')(x)

    x = Flatten(name='flatten')(x)
    x = QDense(
        10,
        kernel_quantizer='quantized_bits(6,0,alpha=1)',
        bias_quantizer='quantized_bits(6,0,alpha=1)',
        name='output_dense',
    )(x)
    x_out = Activation('softmax', name='output_softmax')(x)

    return Model(inputs=[x_in], outputs=[x_out], name='mnist_qkeras_6bit')

### Optional: Train the QKeras Model
This is quantization-aware training. The reduced precision is simulated during training, so the model can adapt to the quantized weights and activations.

Training a quantized model is usually slower than training the floating-point baseline, but it often preserves accuracy better than simply reducing precision after training.

The goal is not just model accuracy. The hardware goal is to reduce multiplier size and possibly move arithmetic away from DSPs into LUTs.

Leave `TRAIN_QKERAS_MODEL = False` for the tutorial. Set it to `True` to regenerate `models/mnist_cnn_qkeras_6bit.keras`.

In [ ]:
TRAIN_QKERAS_MODEL = False
qkeras_model_path = Path('models/mnist_cnn_qkeras_6bit.keras')

if TRAIN_QKERAS_MODEL:
    qkeras_model_path.parent.mkdir(exist_ok=True)
    qmodel = make_qkeras_mnist_cnn()
    qmodel.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    qmodel.fit(
        X_train, y_train_cat,
        batch_size=256, epochs=10,
        validation_split=0.1, shuffle=True,
    )
    qmodel.save(qkeras_model_path)

## Optimizing for resources: Pruning

Pruning sets some trained weights to zero. For FPGA implementation, the value is that zero multiplications can be optimized away, reducing arithmetic resources when the sparsity is visible to the HLS tool.

### Define the Pruning Schedule
Pruning encourages weights to become exactly zero during training. In hardware, this matters because operations like `0 * x` can be optimized away by HLS, reducing arithmetic resources.

The pruning schedule gradually increases sparsity during training instead of zeroing many weights at once. The target here is 50% sparsity, meaning roughly half of the selected layer weights should become zero.

In [ ]:
def make_pruning_config():
    from tensorflow_model_optimization.sparsity import keras as sparsity

    steps_per_epoch = len(X_train) * 9 // 10 // 256
    return {
        'pruning_schedule': sparsity.PolynomialDecay(
            initial_sparsity=0.0,
            final_sparsity=0.50,
            begin_step=steps_per_epoch * 2,
            end_step=steps_per_epoch * 10,
            frequency=steps_per_epoch,
        )
    }


def prune_function(layer):
    import tensorflow_model_optimization as tfmot

    if isinstance(layer, tf.keras.layers.Dense):
        return tfmot.sparsity.keras.prune_low_magnitude(layer, **make_pruning_config())
    return layer

### Optional: Train the Pruned Model
The pruning wrapper is only needed during training. After training, we strip the pruning wrappers and save a normal Keras model with sparse weights.

The default hls4ml tutorials avoid saving only the best checkpoint during pruning, because the best validation-loss epoch may not be the epoch where the target sparsity has been reached.

Leave `TRAIN_PRUNED_MODEL = False` for the tutorial. Set it to `True` to regenerate `models/mnist_cnn_pruned.keras`.

In [ ]:
TRAIN_PRUNED_MODEL = False
pruned_model_path = Path('models/mnist_cnn_pruned.keras')

if TRAIN_PRUNED_MODEL:
    from tensorflow_model_optimization.python.core.sparsity.keras import pruning_callbacks
    from tensorflow_model_optimization.sparsity.keras import strip_pruning

    pruned_model = tf.keras.models.clone_model(make_mnist_cnn(), clone_function=prune_function)
    pruned_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    pruned_model.fit(
        X_train, y_train_cat,
        batch_size=256, epochs=10,
        validation_split=0.1, shuffle=True,
        callbacks=[pruning_callbacks.UpdatePruningStep()],
    )
    pruned_model = strip_pruning(pruned_model)
    pruned_model_path.parent.mkdir(exist_ok=True)
    pruned_model.save(pruned_model_path)

### Inspect Weight Sparsity

The easiest way to see the effect of pruning is to compare the weight distributions before and after pruning. The pruned model should show many weights exactly at zero.

In [ ]:
if 'pruned_model' not in globals():
    pruned_model = load_model(pruned_model_path)

plotting.plot_weight_sparsity(model, pruned_model)

## Compare Pruned and Quantized Designs

This is the hardware payoff for the CNN section. We compare a pruned baseline model converted with `ap_fixed<16,6>` against a QKeras model whose weights and activations were trained at 6-bit precision.

### Load the Comparison Models

In [ ]:
from tensorflow_model_optimization.sparsity.keras import strip_pruning
from tensorflow_model_optimization.python.core.sparsity.keras import pruning_wrapper
from qkeras.utils import _add_supported_quantized_objects

pruned_model_path = Path('models/mnist_cnn_pruned.keras')
qkeras_model_path = Path('models/mnist_cnn_qkeras_6bit.keras')

comparison_custom_objects = {}
_add_supported_quantized_objects(comparison_custom_objects)
comparison_custom_objects['PruneLowMagnitude'] = pruning_wrapper.PruneLowMagnitude

missing_models = [str(path) for path in [pruned_model_path, qkeras_model_path] if not path.exists()]
if missing_models:
    raise FileNotFoundError('Missing prepared comparison model(s): ' + ', '.join(missing_models))

def strip_pruning_if_present(model):
    if any(layer.__class__.__name__ == 'PruneLowMagnitude' for layer in model.layers):
        return strip_pruning(model)
    return model

pruned_model_for_hls = strip_pruning_if_present(
    load_model(pruned_model_path, custom_objects=comparison_custom_objects)
)
quantized_model_for_hls = strip_pruning_if_present(
    load_model(qkeras_model_path, custom_objects=comparison_custom_objects)
)

for label, candidate_model in [
    ('Pruned baseline', pruned_model_for_hls),
    ('Quantized QKeras 6-bit', quantized_model_for_hls),
]:
    y_candidate = candidate_model.predict(X_eval)
    acc = accuracy_score(y_eval, np.argmax(y_candidate, axis=1))
    print(f'{label}: {acc:.4f}')

### Convert Both Models to hls4ml

In [ ]:
def constrain_output_precision(hls_config):
    for layer_name in ['output_dense', 'output']:
        if layer_name in hls_config.get('LayerName', {}):
            hls_config['LayerName'][layer_name].setdefault('Precision', {})['result'] = 'fixed<16,6,RND,SAT>'
    return hls_config

hls_config_pruned = hls4ml.utils.config_from_keras_model(
    pruned_model_for_hls, granularity='name', backend='Vitis', default_precision='ap_fixed<16,6>'
)
hls_config_pruned['Model']['Strategy'] = 'Latency'
hls_config_pruned = constrain_output_precision(hls_config_pruned)

hls_model_pruned = hls4ml.converters.convert_from_keras_model(
    pruned_model_for_hls,
    hls_config=hls_config_pruned,
    backend='Vitis',
    output_dir='mnist_cnn_pruned',
    part='xczu9eg-ffvb1156-2-e',
    io_type='io_stream',
)

hls_config_quantized = hls4ml.utils.config_from_keras_model(
    quantized_model_for_hls, granularity='name', backend='Vitis'
)
hls_config_quantized['Model']['Strategy'] = 'Latency'
hls_config_quantized = constrain_output_precision(hls_config_quantized)

hls_model_quantized = hls4ml.converters.convert_from_keras_model(
    quantized_model_for_hls,
    hls_config=hls_config_quantized,
    backend='Vitis',
    output_dir='mnist_cnn_qkeras_6bit',
    part='xczu9eg-ffvb1156-2-e',
    io_type='io_stream',
)

### Compare Bit-Accurate Emulation

In [ ]:
hls_model_pruned.compile()
hls_model_quantized.compile()

y_pruned_keras = pruned_model_for_hls.predict(X_eval)
y_pruned_hls = hls_model_pruned.predict(X_eval)
y_quantized_keras = quantized_model_for_hls.predict(X_eval)
y_quantized_hls = hls_model_quantized.predict(X_eval)

print('Pruned baseline Keras:  {:.4f}'.format(accuracy_score(y_eval, np.argmax(y_pruned_keras, axis=1))))
print('Pruned baseline hls4ml: {:.4f}'.format(accuracy_score(y_eval, np.argmax(y_pruned_hls, axis=1))))
print('QKeras 6-bit Keras:     {:.4f}'.format(accuracy_score(y_eval, np.argmax(y_quantized_keras, axis=1))))
print('QKeras 6-bit hls4ml:    {:.4f}'.format(accuracy_score(y_eval, np.argmax(y_quantized_hls, axis=1))))

### Synthesize the Comparison Designs

In [ ]:
SYNTHESIZE_COMPARISON_MODELS = False

if SYNTHESIZE_COMPARISON_MODELS:
    hls_model_pruned.build(csim=False)
    hls_model_quantized.build(csim=False)

### Compare Synthesis Reports

In [ ]:
report_candidates = {
    'Pruned baseline': ['mnist_cnn_pruned', 'pruned_cnn'],
    'Quantized QKeras 6-bit': ['mnist_cnn_qkeras_6bit'],
}

def has_csynth_report(project_dir):
    return (Path(project_dir) / 'myproject_prj/solution1/syn/report/myproject_csynth.rpt').exists()

for label, candidates in report_candidates.items():
    report_dir = next((project_dir for project_dir in candidates if has_csynth_report(project_dir)), None)
    if report_dir is None:
        print(f'No synthesis report found for {label}. Checked: {candidates}')
        continue
    print(f'=== {label}: {report_dir} ===')
    hls4ml.report.read_vivado_report(report_dir)

### What to Look For

- **Accuracy**: the Keras and hls4ml numbers should stay close for each model.
- **DSPs**: the 6-bit QKeras design should use fewer DSPs because small multipliers can map to LUTs.
- **LUTs**: LUT use may increase when arithmetic moves out of DSPs.
- **Latency and II**: these should be interpreted together; lower precision mainly changes arithmetic cost, not the graph topology.

## Wrap-Up

The main takeaways are:

- CNNs use `io_stream` in hls4ml.
- Streams map naturally to FIFOs and shift registers for convolution windows.
- Bit-accurate emulation checks fixed-point accuracy before synthesis.
- The synthesis report turns model choices into concrete hardware numbers.
- Quantization and pruning are useful only when the accuracy and resource reports show the tradeoff clearly.

---

## Exercise

Train a new CNN variant combining:
- 8-bit QKeras quantization-aware training.
- Pruning. You can either reuse the current `prune_function()` or modify it to prune the `Conv2D` layers instead.

Convert the trained model to hls4ml, compile it, run bit-accurate emulation, and synthesize it.

In [ ]:
# Solution

def make_qkeras_mnist_cnn_8bit():
    x = x_in = Input(shape=(28, 28, 1))

    x = QConv2DBatchnorm(
        8, (3, 3),
        kernel_quantizer='quantized_bits(8,0,alpha=1)',
        bias_quantizer='quantized_bits(8,0,alpha=1)',
        kernel_initializer='lecun_uniform',
        kernel_regularizer=l1(0.0001),
        use_bias=True,
        name='fused_convbn_0',
    )(x)
    x = QActivation('quantized_relu(8)', name='conv_act_0')(x)
    x = MaxPooling2D((2, 2), name='pool_0')(x)

    x = QConv2DBatchnorm(
        16, (3, 3),
        kernel_quantizer='quantized_bits(8,0,alpha=1)',
        bias_quantizer='quantized_bits(8,0,alpha=1)',
        kernel_initializer='lecun_uniform',
        kernel_regularizer=l1(0.0001),
        use_bias=True,
        name='fused_convbn_1',
    )(x)
    x = QActivation('quantized_relu(8)', name='conv_act_1')(x)
    x = MaxPooling2D((2, 2), name='pool_1')(x)

    x = Flatten(name='flatten')(x)
    x = QDense(
        10,
        kernel_quantizer='quantized_bits(8,0,alpha=1)',
        bias_quantizer='quantized_bits(8,0,alpha=1)',
        name='output_dense',
    )(x)
    x_out = Activation('softmax', name='output_softmax')(x)

    return Model(inputs=[x_in], outputs=[x_out], name='mnist_qkeras_8bit_pruned')

In [ ]:
qmodel_8bit = make_qkeras_mnist_cnn_8bit()
qmodel_8bit_pruned = tf.keras.models.clone_model(qmodel_8bit, clone_function=prune_function)
qmodel_8bit_pruned.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

qmodel_8bit_pruned.fit(
    X_train, y_train_cat,
    batch_size=256, epochs=10,
    validation_split=0.1, shuffle=True,
    callbacks=[pruning_callbacks.UpdatePruningStep()],
)

qmodel_8bit_pruned = strip_pruning(qmodel_8bit_pruned)
qmodel_8bit_pruned.save('models/mnist_cnn_qkeras_8bit_pruned.keras')

In [ ]:
hls_config_8bit_pruned = hls4ml.utils.config_from_keras_model(
    qmodel_8bit_pruned,
    granularity='name',
    backend='Vitis',
)
hls_config_8bit_pruned['Model']['Strategy'] = 'Latency'
hls_config_8bit_pruned = constrain_output_precision(hls_config_8bit_pruned)

hls_model_8bit_pruned = hls4ml.converters.convert_from_keras_model(
    qmodel_8bit_pruned,
    hls_config=hls_config_8bit_pruned,
    backend='Vitis',
    output_dir='mnist_cnn_qkeras_8bit_pruned',
    part='xczu9eg-ffvb1156-2-e',
    io_type='io_stream',
)

In [ ]:
hls_model_8bit_pruned.compile()
y_8bit_pruned_keras = qmodel_8bit_pruned.predict(X_eval)
y_8bit_pruned_hls = hls_model_8bit_pruned.predict(X_eval)

print('8-bit pruned Keras:  {:.4f}'.format(accuracy_score(y_eval, np.argmax(y_8bit_pruned_keras, axis=1))))
print('8-bit pruned hls4ml: {:.4f}'.format(accuracy_score(y_eval, np.argmax(y_8bit_pruned_hls, axis=1))))

In [ ]:
hls_model_8bit_pruned.build(csim=False)

In [ ]:
hls4ml.report.read_vivado_report('mnist_cnn_qkeras_8bit_pruned')